# 🚀 CHẠY AI SERVER TỐC ĐỘ CAO (`Qwen3-8B` + `bge-m3`) TRÊN GOOGLE COLAB CHO BACKEND NESTJS
**Công dụng:** Kéo mô hình `Qwen3-8B` đã được bạn Fine-tune và lưu trên **Hugging Face Hub** về chạy trực tiếp trên GPU Tesla T4 (16GB VRAM) của Google Colab với cấu hình tăng tốc tối đa (**Flash Attention + KV Cache `q8_0` + Pre-warmed VRAM**), đồng thời mở đường truyền HTTPS công khai qua **Cloudflare Tunnel**.

### Các tối ưu hóa tốc độ đã được bật sẵn:
- **`OLLAMA_FLASH_ATTENTION=1`**: Tăng tốc độ đọc thẻ `<context>` RAG dài gấp ~2 lần.
- **`OLLAMA_KV_CACHE_TYPE=q8_0`**: Lượng tử hóa bộ nhớ đệm ngữ cảnh sang 8-bit, tăng tốc độ sinh chữ (tokens/s) trên GPU T4.
- **`OLLAMA_KEEP_ALIVE=24h` & Warm-up**: Nạp sẵn mô hình vào VRAM ngay khi khởi động để câu hỏi đầu tiên của học sinh được trả lời tức thì (~0.8s).

In [ ]:
# BƯỚC 1: Cài đặt zstd, Ollama & Cloudflared trên máy ảo Google Colab
!apt-get update -qq && apt-get install -y -qq zstd pciutils
!curl -fsSL https://ollama.com/install.sh | sh
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
print("✅ Đã cài đặt xong Ollama và Cloudflared!")

In [ ]:
# BƯỚC 2: Khởi động Ollama Server (Flash Attention + KV Cache q8_0), Tạo Modelfile tốc độ cao & Warm-up VRAM
import subprocess, time, os, urllib.request, json

# Điền tên repo Hugging Face chứa mô hình GGUF bạn đã train ở Notebook 1
HF_GGUF_MODEL = "hf.co/tranbateo/qwen3-8b-ai-tutor-vn:Q4_K_M" # Tự động fallback sang "qwen3:8b" nếu repo chưa tồn tại
EMBEDDING_MODEL = "bge-m3"

# Bật bộ tham số tăng tốc tối đa cho GPU Tesla T4 (16GB VRAM)
os.environ["OLLAMA_HOST"] = "0.0.0.0:11434"
os.environ["OLLAMA_ORIGINS"] = "*"
os.environ["OLLAMA_FLASH_ATTENTION"] = "1"
os.environ["OLLAMA_KV_CACHE_TYPE"] = "q8_0"
os.environ["OLLAMA_KEEP_ALIVE"] = "24h"
os.environ["OLLAMA_NUM_PARALLEL"] = "2"

subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)

print(f"⏳ Đang tải mô hình LLM ({HF_GGUF_MODEL}) và Embedding ({EMBEDDING_MODEL}) vào GPU Colab...")
!ollama pull {HF_GGUF_MODEL} && ollama cp {HF_GGUF_MODEL} base-qwen3-tutor || (ollama pull qwen3:8b && ollama cp qwen3:8b base-qwen3-tutor)
!ollama pull {EMBEDDING_MODEL}

# Đóng gói Modelfile tối ưu tốc độ sinh token và ngữ cảnh chuẩn cho AI Tutor
modelfile_content = """FROM base-qwen3-tutor
PARAMETER num_ctx 4096
PARAMETER temperature 0.2
PARAMETER top_p 0.85
PARAMETER top_k 30
PARAMETER repeat_penalty 1.1
"""
with open("Modelfile.tutor", "w", encoding="utf-8") as f:
    f.write(modelfile_content)

!ollama create qwen3-8b-ai-tutor-vn -f Modelfile.tutor

# Warm-up: Nạp sẵn mô hình vào VRAM 16GB để câu hỏi đầu tiên từ Web/Mobile phản hồi tức thì
print("🔥 Đang Warm-up nạp sẵn Qwen3-8B vào VRAM (keep_alive=24h)...")
try:
    req = urllib.request.Request(
        "http://127.0.0.1:11434/api/chat",
        data=json.dumps({
            "model": "qwen3-8b-ai-tutor-vn",
            "messages": [{"role": "user", "content": "Chào thầy cô"}],
            "stream": False,
            "think": False,
            "keep_alive": "24h",
            "options": {"num_predict": 16}
        }).encode("utf-8"),
        headers={"Content-Type": "application/json"}
    )
    urllib.request.urlopen(req, timeout=60)
except Exception as e:
    print(f"Warm-up notice: {e}")

print("✅ Đã nạp sẵn và tối ưu hóa tốc độ cả 2 mô hình trên VRAM của Google Colab!")

In [ ]:
# BƯỚC 3: Mở cổng HTTPS công khai và In cấu hình .env cho Backend NestJS
import re, subprocess, time

tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:11434"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

public_url = None
start_time = time.time()
while time.time() - start_time < 30:
    line = tunnel_proc.stdout.readline()
    match = re.search(r"https://[-a-zA-Z0-9]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        break

if public_url:
    print("=" * 75)
    print("🎉 AI SERVER TỐC ĐỘ CAO CỦA BẠN ĐÃ SẴN SÀNG TRÊN GOOGLE COLAB!")
    print("👉 Hãy copy 4 dòng dưới đây dán vào file code/backend/.env trên laptop:")
    print("-" * 75)
    print(f'AI_PROVIDER="OLLAMA"')
    print(f'OLLAMA_BASE_URL="{public_url}"')
    print(f'OLLAMA_MODEL="qwen3-8b-ai-tutor-vn"')
    print(f'OLLAMA_EMBEDDING_MODEL="bge-m3"')
    print("=" * 75)
else:
    print("⚠️ Không lấy được URL tự động, vui lòng chạy lại ô này.")